# Topic Graph (preview): Better Evidence for Broad Questions

Ask a search engine a **broad** question, such as *"What have space telescopes been used to study?"*, and it returns the chunks **most similar** to the question. Those tend to be near-duplicates from one or two articles, so an agent grounded on them gives a narrow answer and misses most of the story.

A **topic graph** fixes that. It groups your content into **topics** (themes) ahead of time, and at query time Foundry IQ **samples evidence across topics** instead of taking only the closest matches. Same question, same index, same number of chunks, but the evidence covers far more of the collection.

**What you'll see in this notebook**

| | Without a topic graph | With a topic graph |
|---|---|---|
| *"What have space telescopes been used to study?"*, 10 chunks | 7 of the 10 chunks come from the *Hubble* article | The 10 chunks come from ~8 different articles |
| *"What are the major disasters and failures in spaceflight?"* | Evidence mentions about 6 of 14 well-known events | Evidence mentions about 10 of 14 |
| *"Who was the first woman in space?"* (narrow) | *Valentina Tereshkova* is in the top results | Still in the top results: precision isn't lost |

> Your exact numbers will differ slightly; the notebook measures them live.

| | Topic graph | Entity graph ([entity-graph](entity-graph.ipynb)) |
|---|---|---|
| Helps with | Broad questions: *"What are the main themes…?"* | Multi-hop questions: *"Which astronauts launched from sites in Kazakhstan?"* |
| Ontology | Not needed | Required |
| Built by | **You, client-side**, with the Foundry IQ skills plugin | **The service**, via a graph-enabled Blob knowledge source |

![Topic graph: chunk vectors are linked into a similarity graph, grouped into named topics and stored as a topic index; queries use topic_graph none, balanced or diverse](../docs/images/topic-graph.svg)

> ### ⚠️ This notebook has a manual step
> **Step 3 is done by you in GitHub Copilot CLI, not by running cells.** There's no server-side topic graph builder in Foundry IQ (as of November 2026): you build it with the [Foundry IQ skills plugin](../docs/foundry-iq-skills-guide.md). Steps 1–2 run as normal; Step 4 only shows results after Step 3 is finished. Install the tools below **before** you start.

## Prerequisites

| Requirement | Notes |
|---|---|
| [prepare-dataset.ipynb](prepare-dataset.ipynb) completed | Provides the `space-exploration` index. `text_vector` is stored and retrievable with a cosine HNSW profile, as the builder requires |
| Search service with topic graph support in its region | |
| Roles: **Search Service Contributor**, **Search Index Data Contributor**, **Search Index Data Reader** | The plugin requires Microsoft Entra sign-in (`az login`); it doesn't accept API keys |
| GitHub Copilot CLI, Git, Node.js, Python 3.12–3.14, Azure CLI, **Windows x64 or Linux x64/ARM64** | The native topic builder doesn't run on macOS. **On WSL**, Node.js, npm and Azure CLI must be the *Linux* installs, and `libicu` must be installed (see Step 3.1) |
| A chat deployment (`LLM_DEPLOYMENT`) and **Cognitive Services OpenAI User** on the Foundry resource | Names topics during the build, and writes the example answers in Step 4c |


## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv requests matplotlib


In [ ]:
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import requests
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeSourceResultsProcessing,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from azure.search.documents.models import (
    BalancedVectorQueryTopicGraph,
    DiverseVectorQueryTopicGraph,
    NoneVectorQueryTopicGraph,
    VectorizableTextQuery,
)
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
credential = DefaultAzureCredential()
search_credential = AzureKeyCredential(search_api_key) if search_api_key else credential

ai_services_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]
llm_deployment = os.environ["LLM_DEPLOYMENT"]

CONTENT_INDEX = "space-exploration"
VECTOR_FIELD = "text_vector"
KNOWLEDGE_SOURCE = "space-topic-ks"
KNOWLEDGE_BASE = "space-topic-kb"
EVIDENCE_BUDGET = 10  # chunks handed to the answering model, as an agent would

index_client = SearchIndexClient(search_endpoint, search_credential)
content_client = SearchClient(search_endpoint, CONTENT_INDEX, search_credential)

print(f"Search endpoint: {search_endpoint}")
print(f"Answer model:    {llm_deployment}")


## Step 1: The problem — broad questions get narrow evidence

**What we're searching:** the `space-exploration` index from [prepare-dataset](prepare-dataset.ipynb), about 4,000 chunks from ~230 Wikipedia articles on space exploration. Each chunk has its text, its article `title` and an embedding (`text_vector`).

Ask a broad question with a plain **vector search** and look at the 10 chunks an agent would be handed. Vector search ranks each chunk by similarity to the question, so the top results cluster around whichever article is the closest match.


In [ ]:
PROBLEM_QUESTION = "What have space telescopes been used to study?"

plain = list(content_client.search(
    vector_queries=[VectorizableTextQuery(text=PROBLEM_QUESTION, k_nearest_neighbors=EVIDENCE_BUDGET, fields=VECTOR_FIELD)],
    select=["title", "chunk"],
    top=EVIDENCE_BUDGET,
))
titles = Counter(doc["title"] for doc in plain)
print(f"Question: {PROBLEM_QUESTION}\n")
print(f"The top {len(plain)} chunks come from only {len(titles)} articles:")
for title, n in titles.most_common():
    print(f"  {n:>2} × {title}")


Most of the evidence is about one telescope. An agent grounded on it will write a Hubble-centric answer and say little about infrared observatories, X-ray astronomy, exoplanet surveys or star mapping, even though the index covers all of them.

## Step 2: Create the knowledge source and knowledge base

Agents reach Foundry IQ through a **knowledge base**, so that's what we'll compare through in Step 4. Create a `searchIndex` knowledge source over the content index, searching the **vector field**, and a knowledge base on top of it.

Don't set a stored `topic_graph` default yet: the service rejects it until the vector field is linked to a topic index (Step 3). In Step 4 we set `topic_graph` per request instead. The plugin's retrieval setup (Step 3) also expects this knowledge source and knowledge base to exist already.


In [ ]:
# No topic_graph default here: the service rejects it until text_vector has a topicGraphProfile (Step 3).
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE,
    results_processing=KnowledgeSourceResultsProcessing.NONE,
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=CONTENT_INDEX,
        search_fields=[SearchIndexFieldReference(name=VECTOR_FIELD)],
        source_data_fields=[SearchIndexFieldReference(name=f) for f in ("chunk_id", "title", "chunk")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE,
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge source '{KNOWLEDGE_SOURCE}' and knowledge base '{KNOWLEDGE_BASE}' ready")


## Step 3: Build the topic graph (manual, outside the notebook)

> ### 🛑 ACTION REQUIRED: you do this step in a terminal, not by running cells
> Azure AI Search can **use** a topic graph but can't **build** one. You build it on your own machine with the **Foundry IQ skills plugin** for **GitHub Copilot CLI**, then come back here. **Don't continue to Step 4 until the checklist below is complete.**

### What happens in this step

You'll paste two prompts into Copilot CLI. Each one starts a phase that only **reads** first and then stops at an **approval card** before it changes anything in Azure.

| Phase | You paste | The plugin runs | What changes in Azure |
|---|---|---|---|
| **A. Build the topic index** | Prompt 1 | `doctor` → `inspect` → `plan` → *you approve* → `apply` → `verify` | A **new index** named like `fiq-space-exploration-<id>-topics-<id>` is created. The `space-exploration` content index is **not** touched yet |
| **B. Link it for retrieval** | Prompt 2 | `plan-retrieval` → *you approve* → `configure-retrieval` → `verify-retrieval` | `space-exploration` gets a **topic graph profile** (named like `fiq-space-exploration-<id>-retrieval-<id>`) and `text_vector` points to it. The stored default stays `none`, so nothing changes for existing queries until you ask for `balanced` or `diverse` |

**Inside phase A, `apply` does the heavy lifting on your machine:**

1. **Downloads every chunk vector** from `space-exploration` (about 4,000 chunks × 3,072 dimensions). This is why `text_vector` must be *stored* and *retrievable*.
2. **Builds a nearest-neighbour similarity graph** and runs **community detection** at two resolutions, which gives broad **root topics** and narrower **sub-topics**. Our run produced **170 topics: 39 root topics and 131 sub-topics**.
3. **Computes a centroid vector** for each topic.
4. **Names each topic** by sending a few representative chunks to your chat deployment, which returns a title and a description. This is the only billable model usage, and the approval card caps the number of calls.
5. **Uploads** the topics, hierarchy and centroids to the new topic index (about 1,200 rows), then `verify` reads them back.

**Phase B is separate** because building an index doesn't change retrieval. Only the link from `text_vector` to the topic index does, and it touches your live content index, so it gets its own approval.

### ✅ To-do checklist

| | To do | Where | Done when |
|---|---|---|---|
| ☐ | **3.1** Install the tools (once per machine) | Terminal | All checks in 3.1 pass |
| ☐ | **3.2** Run the code cell below to generate your two prompts | This notebook | Prompts are printed with no `<placeholders>` |
| ☐ | **3.3** Start Copilot CLI and install the plugin | Copilot CLI | `/mcp` shows `azure` and `foundry-iq-graph` connected |
| ☐ | **3.4** Paste **Prompt 1**, approve, wait for the build | Copilot CLI | Copilot reports the new generation is **verified** |
| ☐ | **3.5** Paste **Prompt 2**, approve | Copilot CLI | Copilot reports retrieval is **verified** |
| ☐ | **3.6** Come back and run Step 4 | This notebook | Step 4 prints `Topic graph linked` |

---

### 3.1 Install the tools (once per machine)

> 👉 **TO DO:** run each check in a terminal and install anything that's missing.

| Requirement | Check with | Notes |
|---|---|---|
| Windows x64 or Linux x64/ARM64 | `uname -m` | The native builder doesn't run on macOS |
| Git | `git --version` | Needed to add the plugin marketplace |
| Node.js + npm | `node --version` | Used by Copilot CLI and the plugin's MCP servers |
| Python 3.12–3.14 with `venv` | `python3 --version` | The plugin creates its own isolated runtime |
| Azure CLI, signed in with Entra ID | `az account show` | The plugin **doesn't accept API keys** |
| GitHub Copilot CLI | `copilot --version` | Install with `npm install -g @github/copilot` |
| Roles on the search service | — | **Search Service Contributor** and **Search Index Data Contributor** (to create and fill the topic index) |

> **Using WSL?** WSL can silently pick up the **Windows** `node`, `npm` and `az` from `/mnt/c/...`, and then Copilot CLI and the plugin don't work. Run `which node npm az`; none of them should start with `/mnt/c/`. If any do, install the Linux versions:
>
> ```bash
> curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.3/install.sh | bash && source ~/.bashrc
> nvm install --lts                                       # Linux Node.js + npm
> curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash  # Linux Azure CLI
> sudo apt install -y libicu-dev                          # needed by the plugin's Azure MCP server
> hash -r && az login --tenant <tenant-id-or-domain>
> ```

### 3.2 Generate your prompts

> 👉 **TO DO:** run the code cell below this one.

The skill never guesses resource names, so you must give them explicitly. The cell reads your `.env`, looks up the resource group and subscription with the Azure CLI, and prints **Prompt 1** and **Prompt 2** ready to paste. If any value still shows as `<placeholder>`, fix it before continuing.

### 3.3 Start Copilot CLI and install the plugin

> 👉 **TO DO:** in a **new terminal** (not inside this notebook), from the repo folder:

```bash
az login --tenant <tenant-id-or-domain>   # skip if already signed in to the right tenant
copilot                                    # starts the interactive Copilot CLI
```

Then, **inside Copilot CLI** (these are slash commands, not shell commands):

```text
/plugin marketplace add microsoft/azure-skills
/plugin install foundry-iq-skills@azure-skills
```

Type `/exit`, start `copilot` again so the new skills load, then type `/mcp` and check that **`azure`** and **`foundry-iq-graph`** are both connected.

### 3.4 Build the topic index (phase A)

> 👉 **TO DO:** paste **Prompt 1** into Copilot CLI, allow the read-only tool calls, then answer **Approve** on the approval card.

What you'll see:

1. **`doctor`**: readiness checks for the runtime, your sign-in, the search service and quotas. Fix anything it reports before continuing.
2. **`inspect`**: confirms `text_vector` is stored, retrievable and uses a cosine HNSW profile (the dataset notebook set it up that way).
3. **`plan`**: the **approval card** shows the headline, estimated time and cost, and the maximum number of model calls. **Approve** it.
4. **`apply`**: the build described above runs. Expect several minutes.
5. **`verify`**: reads every topic row back and reports the new **generation**.

> If the build stops with `MODEL_CALL_BUDGET_EXHAUSTED`, ask Copilot to raise the model call cap and resume; it continues where it stopped.

### 3.5 Turn on topic retrieval (phase B)

> 👉 **TO DO:** once 3.4 reports **verified**, paste **Prompt 2** into Copilot CLI and **Approve** the second card.

The skill runs `plan-retrieval`, shows the card, then `configure-retrieval` and `verify-retrieval`. The knowledge source `space-topic-ks` and knowledge base `space-topic-kb` from Step 2 must already exist, which is why Step 2 comes first.

### 3.6 Come back here

> 👉 **TO DO:** return to this notebook and run Step 4.

If Step 4 says the topic graph isn't linked, step 3.5 hasn't completed. If a cell hangs right after you sign in again with `az login`, interrupt it and run it again.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `copilot` keeps asking *"Install GitHub Copilot CLI?"* (WSL) | `npm` is the Windows one. Install Linux Node.js (see 3.1), then `npm install -g @github/copilot` |
| `Failed to connect to MCP server "azure"` | On Ubuntu/WSL install `libicu-dev`, then `/exit` and restart `copilot` |
| `SOURCE_VECTORS_NOT_RETRIEVABLE` / `NOT_STORED` | Re-run [prepare-dataset.ipynb](prepare-dataset.ipynb), which creates `text_vector` correctly |
| Copilot can't find the search service | Check the subscription in Prompt 1 matches `az account show` |

More detail: [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md).


In [ ]:
import shutil
import subprocess
from urllib.parse import urlparse

search_service = urlparse(search_endpoint).hostname.split(".")[0]
foundry_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "<foundry-endpoint>").split("/api/projects/")[0]
chat_deployment = os.environ.get("GRAPH_LLM_DEPLOYMENT") or os.environ.get("LLM_DEPLOYMENT", "<chat-deployment>")
embedding_model = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-large")

# Resource group and subscription aren't in .env; look them up with the Azure CLI (current subscription only).
resource_group, subscription = "<resource-group>", "<subscription-id>"
if shutil.which("az"):
    found = subprocess.run(
        ["az", "resource", "list", "--name", search_service,
         "--resource-type", "Microsoft.Search/searchServices", "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True,
    ).stdout.strip()
    if found:
        parts = found.split("/")
        subscription, resource_group = parts[2], parts[4]

print(f"Search service:   {search_service}")
print(f"Resource group:   {resource_group}")
print(f"Subscription:     {subscription}")
print(f"Foundry endpoint: {foundry_endpoint}")
print(f"Chat deployment:  {chat_deployment}")
print(f"Embedding model:  {embedding_model}")
if "<" in resource_group:
    print("\n⚠️ Couldn't find the search service in the current subscription. Run `az account set -s <subscription>` "
          "or replace the placeholders below by hand.")

print(f"""
── Prompt 1: plan and build (paste into Copilot CLI) ──
Check whether this machine is ready to build a Topic Graph. Then inspect the source index `{CONTENT_INDEX}` \
on search service `{search_service}` (resource group `{resource_group}`, subscription `{subscription}`) and plan \
a Topic Graph build. Use key field `chunk_id`, text field `chunk`, source field `source_url`, title field `title`, \
vector field `{VECTOR_FIELD}`, embedding model `{embedding_model}`, and name topics with chat deployment \
`{chat_deployment}` on `{foundry_endpoint}`. Show me the approval card before changing anything.

── Prompt 2: enable retrieval (after the build is verified) ──
Turn on topic retrieval for the verified generation using knowledge source `{KNOWLEDGE_SOURCE}` and knowledge \
base `{KNOWLEDGE_BASE}` on search service `{search_service}`. Keep the stored default off.
""")

## Step 4: See the difference

> **Requires Step 3.** Without it, these cells stop with a message.

Now the payoff. Every comparison below is fair:

- **Same question**, **same index** (`space-exploration`), **same knowledge base** (`space-topic-kb`)
- **Same evidence budget**: the first 10 chunks the knowledge base returns, which is roughly what an agent would put in its prompt
- **Only one thing changes**: the `topic_graph` setting on the request

| `topic_graph` | What the service does |
|---|---|
| `none` | Plain vector search: the 10 closest chunks |
| `balanced` | Gathers a large candidate pool (`min_diversity_pool_size`), then picks chunks that are relevant **and** spread across topics. Recommended default |
| `diverse` | Same, with a stronger push towards different topics (`diversity_factor`, 0–1) |

You never query the topic index yourself: the service uses it behind the scenes to decide which candidates belong to which topic.

### Helpers


In [ ]:
kb_client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=KNOWLEDGE_BASE)

TOPIC_SETTINGS = {
    "none": NoneVectorQueryTopicGraph(),
    "balanced": BalancedVectorQueryTopicGraph(min_diversity_pool_size=1000),
    "diverse": DiverseVectorQueryTopicGraph(diversity_factor=0.5, min_diversity_pool_size=1000),
}


def linked_topic_index() -> tuple[str | None, str | None]:
    """Return (profile name, topic index name) linked to the content index's vector field."""
    content_index = index_client.get_index(CONTENT_INDEX)
    vector_field = next(f for f in content_index.fields if f.name == VECTOR_FIELD)
    profiles = content_index.vector_search.topic_graph_profiles or []
    profile = next((p for p in profiles if p.name == vector_field.topic_graph_profile), None)
    return vector_field.topic_graph_profile, (profile.index_name if profile else None)


def evidence(question: str, setting: str) -> list[dict]:
    """The first EVIDENCE_BUDGET chunks the knowledge base returns for one topic_graph setting."""
    request = KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=question)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
            topic_graph=TOPIC_SETTINGS[setting],
        )],
    )
    refs = kb_client.retrieve(request).references or []
    return [dict(ref.source_data or {}) for ref in refs][:EVIDENCE_BUDGET]


def show_evidence(question: str) -> None:
    """Print, per setting, which articles the evidence comes from."""
    print(f"Question: {question}\n")
    for setting in TOPIC_SETTINGS:
        titles = Counter(chunk["title"] for chunk in evidence(question, setting))
        tally = ", ".join(f"{t} ×{n}" if n > 1 else t for t, n in titles.most_common())
        print(f"{setting:<9} {len(titles):>2} articles │ {tally}\n")


PROFILE_NAME, TOPIC_INDEX = linked_topic_index()
print(f"Topic graph linked: {TOPIC_INDEX}" if TOPIC_INDEX else "⚠️ No topic graph linked yet — complete Step 3 first.")


### 4a. The same question as Step 1, now with topic sampling

Watch the article list. With `none` the 10 chunks pile up on one article; with `balanced` and `diverse` they spread across telescopes, observatories and missions.


In [ ]:
if TOPIC_INDEX:
    show_evidence(PROBLEM_QUESTION)


### 4b. Scoreboard: does the evidence cover what it should?

Article counts show *spread*, but not whether the evidence is *useful*. So for each question we list the things a good answer should mention (a simple **checklist**) and count how many appear anywhere in the 10 chunks.

The last question is deliberately **narrow**. It checks the obvious worry: *does spreading the evidence hurt precise questions?* The expected answer should still be there with every setting.


In [ ]:
CHECKLISTS = {
    "What have space telescopes been used to study?": {
        "Hubble": r"Hubble", "James Webb": r"Webb|JWST", "Spitzer": r"Spitzer", "Chandra": r"Chandra",
        "Kepler": r"Kepler", "TESS": r"TESS", "Gaia": r"Gaia", "Herschel": r"Herschel", "Planck": r"Planck",
        "exoplanets": r"exoplanet", "cosmic microwave background": r"cosmic microwave", "dark energy": r"dark energy",
    },
    "What are the major disasters and failures in the history of spaceflight?": {
        "Apollo 1": r"Apollo 1\b", "Apollo 13": r"Apollo 13", "Challenger": r"Challenger", "Columbia": r"Columbia",
        "Soyuz 1": r"Soyuz 1\b", "Soyuz 11": r"Soyuz 11", "Nedelin catastrophe": r"Nedelin", "Starship tests": r"Starship",
        "Mars Climate Orbiter": r"Climate Orbiter", "Mars Polar Lander": r"Polar Lander", "Antares": r"Antares",
        "Falcon 9": r"Falcon 9", "Beagle 2": r"Beagle", "Phobos probes": r"Phobos",
    },
    "What dangers do astronauts face in space?": {
        "radiation": r"radiation", "bone loss": r"\bbone", "muscle loss": r"muscle", "vision problems": r"vision|\beye",
        "fire": r"\bfire", "space debris": r"debris", "launch and re-entry accidents": r"Challenger|Columbia|Soyuz 11|Apollo 1\b",
        "isolation": r"psycholog|isolation", "spacewalk hazards": r"spacewalk|extravehicular", "decompression": r"decompress|depressur",
    },
    "Who was the first woman in space?": {"Valentina Tereshkova": r"Tereshkova"},
}


def covered(text: str, checklist: dict) -> list[str]:
    return [item for item, pattern in checklist.items() if re.search(pattern, text, re.IGNORECASE)]


def scoreboard() -> dict:
    scores = {}
    print(f"{'question':<48} {'setting':<9} {'articles':>8}   checklist items in evidence")
    for question, checklist in CHECKLISTS.items():
        for setting in TOPIC_SETTINGS:
            chunks = evidence(question, setting)
            hits = covered(" ".join(c["chunk"] for c in chunks), checklist)
            scores[(question, setting)] = len(hits) / len(checklist)
            label = question[:46] if setting == "none" else ""
            print(f"{label:<48} {setting:<9} {len({c['title'] for c in chunks}):>8}   {len(hits):>2}/{len(checklist)}")
        print()
    return scores


def plot_scores(scores: dict) -> None:
    questions = list(CHECKLISTS)
    width = 0.27
    plt.figure(figsize=(11, 4))
    for i, setting in enumerate(TOPIC_SETTINGS):
        values = [100 * scores[(q, setting)] for q in questions]
        plt.bar([x + (i - 1) * width for x in range(len(questions))], values, width, label=setting)
    plt.xticks(range(len(questions)), [q[:32] + "…" for q in questions], fontsize=8)
    plt.ylabel("checklist covered by the 10 chunks (%)")
    plt.ylim(0, 105)
    plt.legend(title="topic_graph")
    plt.tight_layout()
    plt.show()


if TOPIC_INDEX:
    plot_scores(scoreboard())


### 4c. What the user actually reads: the written answer

Evidence is only half the story; users read the **answer**. Below, the same chat model writes an answer from each set of 10 chunks, with the same prompt. The only difference is the evidence it was given, so any difference in the answers comes from retrieval.

> We call the chat model directly so the comparison is isolated. In production an agent (or a knowledge base with `answerSynthesis` output) does this step for you.


In [ ]:
def write_answer(question: str, chunks: list[dict]) -> str:
    """Have the chat deployment answer from the given chunks only."""
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    sources = "\n\n".join(f"[{c['title']}] {c['chunk'][:800]}" for c in chunks)
    response = requests.post(
        f"{ai_services_endpoint}/openai/deployments/{llm_deployment}/chat/completions?api-version=2024-10-21",
        headers={"Authorization": f"Bearer {token}"},
        json={"messages": [
            {"role": "system", "content": "Answer only from the sources, in 5-8 short bullet points."},
            {"role": "user", "content": f"Question: {question}\n\nSources:\n{sources}"},
        ]},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def compare_answers(question: str, settings=("none", "diverse")) -> None:
    checklist = CHECKLISTS[question]
    print(f"Question: {question}\n")
    for setting in settings:
        answer = write_answer(question, evidence(question, setting))
        hits = covered(answer, checklist)
        print(f"{'═' * 20} topic_graph = {setting}: answer mentions {len(hits)}/{len(checklist)} checklist items {'═' * 20}")
        print(answer)
        print(f"\n✔ {', '.join(hits) or '—'}\n")


if TOPIC_INDEX:
    compare_answers("What are the major disasters and failures in the history of spaceflight?")


### What to take away

- **Evidence spreads out.** With `none`, broad questions pull most chunks from one or two articles (7 of 10 from *Hubble*). With `balanced`/`diverse`, the same 10 slots cover two to three times more articles
- **More of the story gets in.** On the disasters question, the evidence goes from about 6 to 10 of 14 well-known events, and the written answer picks up Apollo 13 and the Phobos probe failure that the `none` answer never mentions
- **More isn't always better.** Push diversity too far and some slots go to loosely related chunks (in our run, `diverse` swapped telescope chunks for NASA field-centre pages). That's why `balanced` is the recommended default and `diverse` is tunable
- **Narrow questions are safe.** The first-woman-in-space answer is found with every setting
- **No extra model calls at query time.** Topic sampling only changes *which* candidates are kept, so latency and cost stay close to plain vector search

## When should you use a topic graph?

| Use it for | Not needed for |
|---|---|
| Broad or exploratory questions: *"What are the main themes…?"*, *"Give me an overview of…"* | Precise lookups: *"When did Sputnik 1 launch?"* (plain or agentic search is enough) |
| Summaries, briefings and reports that should cover the whole collection | Multi-hop relationship questions: use an [entity graph](entity-graph.ipynb) |
| Corpora with **many near-duplicate chunks** (long articles, repeated boilerplate, many versions of a document) | Small or already diverse corpora, where the closest chunks are varied anyway |

**Which setting?** Start with `balanced` as the stored default on the knowledge source. Use `diverse` per request when the user explicitly asks for breadth, and tune `diversity_factor` on your own questions.


## Under the hood (optional): what a topic graph is made of

You don't need this section to use topic graphs. It explains how the plugin builds one and lets you look inside the result.

### How it's built

1. **Similarity graph:** each chunk is a node, connected to its nearest neighbours by vector similarity
2. **Community detection:** densely connected groups of chunks become **topics**. Running detection at several resolutions gives a **hierarchy** (broad root topics → narrower sub-topics, at most 100 chunks per leaf)
3. **Centroids:** each topic gets a representative vector, the normalised mean of its chunks' vectors
4. **Naming:** an LLM reads a few chunks closest to each centroid and gives the topic a short title and a one-sentence description

### How it's stored

The topic index is a small graph of **subject → predicate → object** rows (one row per fact, not one document per topic):

| Predicate | Meaning | Example |
|---|---|---|
| `HasRootTopic` | The graph's top-level topics | `topicgraph-topic` → `Topic 7` |
| `HasSubTopic` | Parent → child in the hierarchy | `Topic 7` → `Topic 60` (*Falcon 9 Launch Vehicle*) |
| `HasCentroidVector` | The topic's centroid, used at query time | `Topic 60` → a 3,072-dimension vector |
| `prop:title`, `prop:description` | LLM-generated name and summary | *SpaceX and Reusable Launch Systems* |
| `prop:level`, `prop:proportion`, `prop:parentProportion` | Depth in the hierarchy and share of the corpus | `level = 0`, `parentProportion ≈ 0.05` |

The content index's `text_vector` field points to the topic index through a **`topic_graph_profile`**; that link is what Step 3.5 created.

### Inspection helpers


In [ ]:
from collections import defaultdict


def describe_topic_index(name: str, sample: int = 5) -> None:
    topic_index = index_client.get_index(name)
    print(f"Topic index '{name}' fields:")
    for field in topic_index.fields:
        dims = f"dims={field.vector_search_dimensions}" if field.vector_search_dimensions else ""
        print(f"  {field.name:<26} {str(field.type):<36} {dims}")
    results = SearchClient(search_endpoint, name, search_credential).search(search_text="*", include_total_count=True, top=sample)
    rows = list(results)
    print(f"\n{results.get_count()} rows (subject → predicate → object facts). Sample, empty fields hidden:")
    for doc in rows:
        print({k: v for k, v in doc.items() if not k.startswith("@") and k != "id" and v not in (None, "", [])})


def show_topic_tree(name: str, max_children: int = 4) -> None:
    rows = list(SearchClient(search_endpoint, name, search_credential).search(
        search_text="*",
        filter="predicate eq 'HasRootTopic' or predicate eq 'HasSubTopic'",
        select=["subject_id", "predicate", "object_id", "object_id_display_name"],
        top=1000,
    ))
    roots = [r for r in rows if r["predicate"] == "HasRootTopic"]
    children = defaultdict(list)
    for r in rows:
        if r["predicate"] == "HasSubTopic":
            children[r["subject_id"]].append(r["object_id_display_name"])
    print(f"{len(roots)} root topics, {sum(map(len, children.values()))} sub-topics. Largest root topics first:\n")
    for root in sorted(roots, key=lambda r: -len(children[r["object_id"]])):
        subs = children[root["object_id"]]
        more = f", … (+{len(subs) - max_children})" if len(subs) > max_children else ""
        print(f"• {root['object_id_display_name']}")
        if subs:
            print(f"    └ {', '.join(subs[:max_children])}{more}")


### Look inside: the link, the raw rows and the topic tree


In [ ]:
PROFILE_NAME, TOPIC_INDEX = linked_topic_index()
print(f"{VECTOR_FIELD}.topic_graph_profile = {PROFILE_NAME}")
print(f"Linked topic index: {TOPIC_INDEX or 'none yet — complete Step 4 first'}")

In [ ]:
if TOPIC_INDEX:
    describe_topic_index(TOPIC_INDEX)

In [ ]:
if TOPIC_INDEX:
    show_topic_tree(TOPIC_INDEX)


## Clean up

Delete the knowledge base and knowledge source created here. Retire the topic index through the plugin (*"retire my topic graph staging"*) so its own state stays consistent; deleting it by hand would leave the content index's `topic_graph_profile` pointing at a missing index.


In [ ]:
# index_client.delete_knowledge_base(KNOWLEDGE_BASE)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge base and knowledge source")

## Summary

- **The problem:** for broad questions, plain vector search hands an agent near-duplicate chunks from one or two articles, so the answer is narrow
- **The fix:** a topic graph lets Foundry IQ sample evidence across themes. Same question, same 10 slots, several times more articles and more of what a good answer should cover, with no extra model calls at query time
- **The cost:** a one-off, client-side build with the Foundry IQ skills plugin (two approved phases: build the topic index, then link it to `text_vector`)
- **How to use it:** set `topic_graph` on the knowledge source (stored default) or per request: `none`, `balanced` (recommended) or `diverse`

### Next steps

- [entity-graph.ipynb](entity-graph.ipynb): multi-hop relationship questions with an entity graph
- [microsoft-retriever-1.ipynb](microsoft-retriever-1.ipynb): multi-part questions with Microsoft Retriever 1
- [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md): the full plugin workflow
